# Concordance 
* 391_concordance_SAT_ID_ref_stockholmarchipelagotrail.ipynb

In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-09-15 02:33


In [2]:
import requests
from urllib.parse import quote
import pandas as pd

CONCORDANCE_URL = (
    "https://map.stockholmarchipelagotrail.com/"
    "data/geojson/poi-concordance.json"
)

API_URL = (
    "https://map.stockholmarchipelagotrail.com/"
    "api/objects/"
)

session = requests.Session()


# ============================================================
# 1. Read concordance
# ============================================================

data = session.get(
    CONCORDANCE_URL,
    timeout=30
).json()

sat_id_of = data["satIdOf"]

sat_ids = list(sat_id_of.values())

print(f"SAT objects: {len(sat_ids)}")


# ============================================================
# 2. Query API
# ============================================================

api_objects = {}

for i, sat_id in enumerate(sat_ids, 1):

    url = API_URL + quote(sat_id, safe="")

    try:
        response = session.get(
            url,
            timeout=20
        )

        if response.status_code == 200:
            api_objects[sat_id] = response.json()

        else:
            api_objects[sat_id] = {
                "_api_error": response.status_code
            }

    except Exception as e:

        api_objects[sat_id] = {
            "_api_error": str(e)
        }

    if i % 100 == 0 or i == len(sat_ids):
        print(f"{i}/{len(sat_ids)}")

print("API check complete")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


SAT objects: 1143
100/1143
200/1143
300/1143
400/1143
500/1143
600/1143
700/1143
800/1143
900/1143
1000/1143
1100/1143
1143/1143
API check complete


In [3]:
# ============================================================
# 3. Status distribution
# ============================================================

statuses = pd.Series([
    obj.get("status")
    for obj in api_objects.values()
])

print(
    statuses
    .fillna("(no status)")
    .value_counts()
)

(no status)    783
missing         38
Name: count, dtype: int64


In [4]:
# ============================================================
# 4. Inspect the two known examples
# ============================================================

for sat_id in [
    "sat:poi:2dgqk",
    "sat:poi:237tx"
]:

    obj = api_objects.get(sat_id, {})

    print("\n" + "=" * 60)
    print(sat_id)

    print("status:",
          obj.get("status"))

    print("firstSeen:",
          obj.get("firstSeen"))

    print("alternateIdentifiers:",
          obj.get("alternateIdentifiers"))

    print("sameAs:",
          obj.get("sameAs"))

    print("wikidata:",
          obj.get("wikidata"))

    print("error:",
          obj.get("_api_error"))


sat:poi:2dgqk
status: None
firstSeen: 2026-08-24
alternateIdentifiers: ['osm:way:316767169']
sameAs: ['https://www.openstreetmap.org/way/316767169']
wikidata: None
error: None

sat:poi:237tx
status: missing
firstSeen: 2026-06-10
alternateIdentifiers: ['osm:way:857838618', 'wikidata:Q133864018']
sameAs: None
wikidata: None
error: None


In [7]:
def extract_links(obj):

    osm = []
    wikidata = []

    # --------------------------------------------------------
    # alternateIdentifiers
    # --------------------------------------------------------

    for identifier in obj.get(
        "alternateIdentifiers", []
    ) or []:

        identifier = str(identifier).strip()

        if identifier.startswith("osm:"):

            parts = identifier.split(":")

            if len(parts) == 3:

                osm_type = parts[1]
                osm_id = parts[2]

                if osm_type in [
                    "node",
                    "way",
                    "relation"
                ]:

                    osm.append(
                        f"[{osm_type}:{osm_id}]"
                        f"(https://www.openstreetmap.org/"
                        f"{osm_type}/{osm_id})"
                    )

        elif identifier.startswith("wikidata:"):

            qid = identifier.split(":", 1)[1]

            wikidata.append(
                f"[{qid}]"
                f"(https://www.wikidata.org/wiki/{qid})"
            )

    # --------------------------------------------------------
    # sameAs
    # --------------------------------------------------------

    for link in obj.get("sameAs", []) or []:

        link = str(link).strip()

        if "openstreetmap.org/" in link:
            osm.append(f"[OSM]({link})")

        elif "wikidata.org/wiki/" in link:

            qid = link.rstrip("/").split("/")[-1]

            wikidata.append(
                f"[{qid}]({link})"
            )

    # Remove duplicates
    osm = list(dict.fromkeys(osm))
    wikidata = list(dict.fromkeys(wikidata))

    return "<br>".join(osm), "<br>".join(wikidata)

In [8]:
# ============================================================
# 3. Build Markdown table
# ============================================================

rows = []

for sat_id, obj in api_objects.items():

    sat_link = (
        f"[{sat_id}]"
        f"(https://map.stockholmarchipelagotrail.com/"
        f"?poi={quote(sat_id, safe='')})"
    )

    json_link = (
        f"[JSON]"
        f"(https://map.stockholmarchipelagotrail.com/"
        f"api/objects/{quote(sat_id, safe='')})"
    )

    osm, wikidata = extract_links(obj)

    rows.append({
        "SAT ID": sat_link,
        "JSON": json_link,
        "Status": obj.get("status", ""),
        "firstSeen": obj.get("firstSeen", ""),
        "OSM": osm,
        "Wikidata": wikidata
    })


result = pd.DataFrame(rows)

In [9]:
print(
    result["Status"]
    .replace("", "(no status)")
    .value_counts()
)

Status
(no status)    783
missing         38
Name: count, dtype: int64


In [12]:
from IPython.display import display, HTML
from html import escape
import pandas as pd


def get_osm_url(obj):
    # 1. API:s explicita OSM-länk
    links = obj.get("links", {})
    if links.get("openstreetmap"):
        return links["openstreetmap"]

    # 2. sameAs
    for url in obj.get("sameAs", []):
        if "openstreetmap.org/" in url:
            return url

    # 3. alternateIdentifiers
    for identifier in obj.get("alternateIdentifiers", []):
        identifier = str(identifier)

        if identifier.startswith("osm:node:"):
            return f"https://www.openstreetmap.org/node/{identifier.split(':')[-1]}"

        if identifier.startswith("osm:way:"):
            return f"https://www.openstreetmap.org/way/{identifier.split(':')[-1]}"

        if identifier.startswith("osm:relation:"):
            return f"https://www.openstreetmap.org/relation/{identifier.split(':')[-1]}"

    return None


def get_osm_label(url):
    if not url:
        return ""

    parts = url.rstrip("/").split("/")
    if len(parts) >= 2:
        object_type = parts[-2]
        object_id = parts[-1]
        return f"{object_type} {object_id}"

    return "OSM"


def get_wikidata_url(obj):
    # Explicit wikidata link
    links = obj.get("links", {})
    if links.get("wikidata"):
        return links["wikidata"]

    # sameAs
    for url in obj.get("sameAs", []):
        if "wikidata.org/wiki/" in url:
            return url

    # wikidata field
    qid = obj.get("wikidata")
    if qid:
        return f"https://www.wikidata.org/wiki/{qid}"

    # alternateIdentifiers
    for identifier in obj.get("alternateIdentifiers", []):
        identifier = str(identifier)

        if identifier.startswith("wikidata:"):
            qid = identifier.split(":", 1)[1]
            return f"https://www.wikidata.org/wiki/{qid}"

    return None


def get_wikidata_label(url):
    if not url:
        return ""

    return url.rstrip("/").split("/")[-1]


def make_link(label, url):
    if not url:
        return ""

    return (
        f'<a href="{escape(url)}" target="_blank">'
        f'{escape(label)}</a>'
    )  
result["OSM"] = result["object"].apply(
    lambda x: make_link(
        get_osm_label(get_osm_url(x)),
        get_osm_url(x)
    )
)

result["Wikidata"] = result["object"].apply(
    lambda x: make_link(
        get_wikidata_label(get_wikidata_url(x)),
        get_wikidata_url(x)
    )
) 
display(
    HTML(
        result[
            ["id", "name", "category", "source", "OSM", "Wikidata"]
        ].to_html(
            escape=False,
            index=False
        )
    )
)

KeyError: 'object'

In [23]:
import json
import pandas as pd
from IPython.display import display, HTML
from html import escape


# ============================================================
# Parse JSON
# ============================================================

def parse_json(x):
    if pd.isna(x):
        return {}

    if isinstance(x, dict):
        return x

    try:
        return json.loads(x)
    except (json.JSONDecodeError, TypeError):
        return {}


# ============================================================
# OSM
# ============================================================

def get_osm_url(obj):

    links = obj.get("links", {})

    # 1. API:s explicita OSM-länk
    if links.get("openstreetmap"):
        return links["openstreetmap"]

    # 2. sameAs
    for url in obj.get("sameAs", []):
        if "openstreetmap.org/" in url:
            return url

    # 3. alternateIdentifiers
    for identifier in obj.get("alternateIdentifiers", []):
        identifier = str(identifier)

        if identifier.startswith("osm:node:"):
            return (
                "https://www.openstreetmap.org/node/"
                + identifier.split(":")[-1]
            )

        if identifier.startswith("osm:way:"):
            return (
                "https://www.openstreetmap.org/way/"
                + identifier.split(":")[-1]
            )

        if identifier.startswith("osm:relation:"):
            return (
                "https://www.openstreetmap.org/relation/"
                + identifier.split(":")[-1]
            )

    return None


def get_osm_label(url):

    if not url:
        return ""

    parts = url.rstrip("/").split("/")

    if len(parts) >= 2:
        return f"{parts[-2]} {parts[-1]}"

    return "OSM"


# ============================================================
# Wikidata
# ============================================================

def get_wikidata_url(obj):

    links = obj.get("links", {})

    # 1. API:s explicita Wikidata-länk
    if links.get("wikidata"):
        return links["wikidata"]

    # 2. sameAs
    for url in obj.get("sameAs", []):
        if "wikidata.org/wiki/" in url:
            return url

    # 3. wikidata-fält
    qid = obj.get("wikidata")

    if qid:
        return f"https://www.wikidata.org/wiki/{qid}"

    # 4. alternateIdentifiers
    for identifier in obj.get("alternateIdentifiers", []):
        identifier = str(identifier)

        if identifier.startswith("wikidata:"):
            qid = identifier.split(":", 1)[1]
            return f"https://www.wikidata.org/wiki/{qid}"

    return None


def get_wikidata_label(url):

    if not url:
        return ""

    return url.rstrip("/").split("/")[-1]


# ============================================================
# HTML link
# ============================================================

def html_link(label, url):

    if not url:
        return ""

    return (
        f'<a href="{escape(url)}" target="_blank">'
        f'{escape(label)}'
        f'</a>'
    )


# ============================================================
# SAT permalink
# ============================================================

def get_sat_url(sat_id):

    if pd.isna(sat_id):
        return ""

    return (
        "https://map.stockholmarchipelagotrail.com/"
        "?poi="
        + str(sat_id).replace(":", "%3A")
    )


# ============================================================
# Parse API JSON
# ============================================================

result["_obj"] = result["JSON"].apply(parse_json)


# ============================================================
# Create HTML columns
# ============================================================

result["OSM"] = result["_obj"].apply(
    lambda obj: html_link(
        get_osm_label(get_osm_url(obj)),
        get_osm_url(obj)
    )
)

result["Wikidata"] = result["_obj"].apply(
    lambda obj: html_link(
        get_wikidata_label(get_wikidata_url(obj)),
        get_wikidata_url(obj)
    )
)


# ============================================================
# HTML version of SAT ID
# ============================================================

result["_SAT_HTML"] = result["SAT ID"].apply(
    lambda x: html_link(
        x,
        get_sat_url(x)
    )
)


# ============================================================
# 1. HTML TABLE — ALL
# ============================================================

html_all = result[
    ["_SAT_HTML", "Status", "firstSeen", "OSM", "Wikidata"]
].rename(
    columns={"_SAT_HTML": "SAT ID"}
)

display(
    HTML(
        "<h3>Alla SAT-objekt</h3>"
        + html_all.to_html(
            escape=False,
            index=False
        )
    )
)


# ============================================================
# 2. HTML TABLE — STATUS = MISSING
# ============================================================

missing = result[
    result["Status"].fillna("").str.lower() == "missing"
].copy()

html_missing = missing[
    ["_SAT_HTML", "Status", "firstSeen", "OSM", "Wikidata"]
].rename(
    columns={"_SAT_HTML": "SAT ID"}
)

display(
    HTML(
        "<h3>Status = missing</h3>"
        + html_missing.to_html(
            escape=False,
            index=False
        )
    )
)



SAT ID,Status,firstSeen,OSM,Wikidata
[sat:poi:eb5j5](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Aeb5j5),,2026-06-23,,
[sat:poi:cc37k](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Acc37k),,2026-06-10,,
[sat:poi:fqw8k](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Afqw8k),,2026-06-10,,
[sat:poi:459zw](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3A459zw),,2026-06-10,,
[sat:poi:4f4rw](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3A4f4rw),,2026-06-10,,
[sat:poi:7dvfp](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3A7dvfp),,2026-06-15,,
[sat:poi:2hhpt](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3A2hhpt),,2026-08-31,,
[sat:poi:pd56m](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Apd56m),,2026-06-15,,
[sat:poi:t6rmc](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3At6rmc),,2026-06-10,,
[sat:poi:kndjy](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Akndjy),,2026-06-15,,


SAT ID,Status,firstSeen,OSM,Wikidata
[sat:poi:ghgfg](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Aghgfg),missing,2026-06-10,,
[sat:poi:8kxjh](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3A8kxjh),missing,2026-06-10,,
[sat:poi:eqe2j](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Aeqe2j),missing,2026-06-26,,
[sat:poi:9xbg8](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3A9xbg8),missing,2026-06-26,,
[sat:poi:f57he](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Af57he),missing,2026-06-26,,
[sat:poi:a3nx8](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Aa3nx8),missing,2026-06-10,,
[sat:poi:5qbbt](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3A5qbbt),missing,2026-06-10,,
[sat:poi:hpy7b](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Ahpy7b),missing,2026-06-10,,
[sat:poi:nmey8](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Anmey8),missing,2026-06-10,,
[sat:poi:rcx5k](https://map.stockholmarchipelagotrail.com/?poi=sat%3Apoi%3Arcx5k),missing,2026-06-26,,


In [17]:
print(type(result["JSON"].iloc[0]))

<class 'str'>
